In [6]:
import pandas as pd
import requests
import json
from utlis import get_api_entry_by_llm, get_data, merge_by_qid
from question_classify import classify

In [8]:
classified_val_data = classify("data/val.json","LLM small")
classified_test_data = classify("data/test.json","LLM small")


KeyError: 'choices'

In [9]:
classified_test_1_data = classify("data/test_1.json","LLM small")

KeyError: 'choices'

In [ ]:
with open("data/classified_val.json", "w", encoding="utf-8") as f:
    json.dump(classified_val_data, f, ensure_ascii=False, indent=4)
with open("data/classified_test.json", "w", encoding="utf-8") as f:
    json.dump(classified_test_data, f, ensure_ascii=False, indent=4)



In [7]:
with open("data/classified_test_1.json", "w", encoding="utf-8") as f:
    json.dump(classified_test_1_data, f, ensure_ascii=False, indent=4)

# STEM

In [7]:
with open('data/classified_val.json', 'r',encoding="utf-8") as file:
    classified_val_data = json.load(file)

In [8]:
stem_data = []
for data in classified_val_data:
    if data['label']=="STEM":
        stem_data.append(data)

In [9]:
for data in stem_data:
    print(f"{data['qid']}. {data['answer']}\n")

val_0004. B

val_0005. C

val_0012. B

val_0013. D

val_0016. B

val_0020. C

val_0023. A

val_0024. A

val_0025. B

val_0026. B

val_0027. C

val_0033. C

val_0038. B

val_0040. E

val_0042. B

val_0043. D

val_0047. A

val_0049. B

val_0051. A

val_0059. A

val_0064. A

val_0065. B

val_0066. E

val_0069. A

val_0071. C

val_0076. D

val_0078. A

val_0082. B

val_0084. D

val_0085. A

val_0087. B

val_0090. B

val_0091. A

val_0093. D



In [10]:
import re

def extract_answers(model_output: str):
    """
    Extract answers from model output following pattern:
    ANSWER[qid]=X
    Returns: List of dicts: [{"qid": "...", "answer": "A"}, ...]
    """

    # 1. Tìm block giữa BEGIN/END ANSWERS (nếu có)
    match_block = re.search(
        r"---BEGIN ANSWERS---(.*?)---END ANSWERS---",
        model_output,
        flags=re.DOTALL | re.IGNORECASE
    )

    if match_block:
        answer_block = match_block.group(1)
    else:
        # fallback: xử lý luôn toàn bộ output
        answer_block = model_output  

    # 2. Regex bắt dòng theo format ANSWER[qid]=X
    pattern = r"ANSWER\[(.*?)\]=([A-Z])"

    matches = re.findall(pattern, answer_block)

    # 3. Convert to structured list
    result = [{"qid": qid.strip(), "answer": ans.strip()} for qid, ans in matches]

    return result


In [24]:
system_prompt = '''
PHASE 1 — EXTERNAL REASONING (HIỂN THỊ BƯỚC GIẢI)
====================================================
Bạn là mô hình chuyên gia giải các bài toán STEM (Toán, Lý, Hóa, Sinh, Thống kê, Công nghệ, Kinh tế học kỹ thuật).

Đối với MỖI câu hỏi trong danh sách đầu vào, bạn phải:
1. Đọc nội dung câu hỏi.
2. Đọc danh sách choices (mảng không có A/B/C/D).
3. Gán nhãn vị trí cho từng lựa chọn:
      choice[0] → A
      choice[1] → B
      choice[2] → C
      choice[3] → D
      ...
4. Giải bài toán theo trình tự rõ ràng:
   (a) Xác định dữ kiện và yêu cầu cần tìm.
   (b) Gọi tên công thức hoặc định luật phù hợp.
   (c) Thay số, biến đổi, rút gọn, kiểm tra sai số.
   (d) Tính ra kết quả cuối cùng (dạng số hoặc biểu thức).
   (e) So sánh kết quả thu được với từng lựa chọn.
   (f) Xác định lựa chọn đúng theo vị trí (A/B/C/D/...).

Bạn được phép:
- Hiển thị toàn bộ chain-of-thought, tính toán, lập luận, công thức.
- Dùng LaTeX để biểu diễn công thức.

KHÔNG ĐƯỢC:
- Nhảy thẳng tới đáp án mà không giải thích.
- Bỏ qua bước so sánh với từng lựa chọn.

SAU KHI HOÀN THÀNH PHẦN GIẢI CỦA TẤT CẢ CÂU HỎI,
bạn phải CHUYỂN sang PHASE 2 và CHỈ TRẢ VỀ JSON ARRAY DUY NHẤT theo format yêu cầu.

====================================================
PHASE 2 — FINAL OUTPUT (CHỈ JSON ARRAY)
=========================================
Trong phase này, bạn phải TRẢ VỀ DUY NHẤT một JSON array.

Mỗi phần tử phải có dạng:
{
  "qid": "<qid>",
  "answer": "<A|B|C|D|E|...>"
}

YÊU CẦU BẮT BUỘC:
- KHÔNG được ghi bất kỳ văn bản, nhãn phase, giải thích hay ký tự nào trước hoặc sau JSON array.
- Chỉ được xuất đúng một JSON array chứa đúng số lượng câu hỏi trong user prompt.
- answer phải là A/B/C/D/E/... dựa theo **vị trí** của lựa chọn đúng.
- KHÔNG được in lại nội dung đáp án, chỉ in chữ cái.
- KHÔNG được in chain-of-thought trong Phase 2.
- KHÔNG được in văn bản ngoài JSON (nếu có → sai format).

Ví dụ hợp lệ:
[
  {"qid": "q1", "answer": "C"},
  {"qid": "q2", "answer": "A"},
  {"qid": "q3", "answer": "D"}
]
'''


In [25]:
vnpt_model_large = get_api_entry_by_llm("LLM large")

In [26]:
results = []
for i in range(0, len(stem_data), 5):
    question_str = ""
    if i + 5>len(stem_data):
        question_str = "\n\n".join([f"{stem_data[j]['qid']}. {stem_data[j]['question']} \n {stem_data[j]['choices']}" for j in range(i, len(stem_data))]) 
    else:
        question_str = "\n\n".join([f"{stem_data[i+j]['qid']}. {stem_data[i+j]['question']} \n {stem_data[i+j]['choices']}" for j in range(5)])
    user_prompt = f'''
        Danh sách các câu hỏi:
        {question_str}
        '''
    headers = { 
            'Authorization': vnpt_model_large["authorization"], 
            'Token-id': vnpt_model_large["tokenId"], 
            'Token-key': vnpt_model_large["tokenKey"], 
            'Content-Type': 'application/json', 
        }
    json_data = {
            'model': 'vnptai_hackathon_large', 
            'messages': [
                {'role': 'system', 'content': system_prompt},
                {'role': 'user', 'content': user_prompt},
            ],
            'temperature': 0.0, 
            'top_p': 1.0, 
            'top_k': 0, 
            'n': 1, 
            'max_completion_tokens': 2048,
        }
    endpoint = "/v1/chat/completions/vnptai-hackathon-large"
    response = requests.post(f'https://api.idg.vnpt.vn/data-service{endpoint}', headers=headers, json=json_data) 
    response_js = response.json()
    print(response_js["choices"][0]["message"]["content"])
    # result = extract_answers(response_js["choices"][0]["message"]["content"])
    # results.extend(result)
    

PHASE 1 — EXTERNAL REASONING

### val_0004: Độ co giãn của cầu theo giá

(a) Xác định dữ kiện và yêu cầu cần tìm:
- Giá ban đầu: $P_1 = 2,00$ đô la
- Giá mới: $P_2 = 2,50$ đô la
- Lượng cầu ban đầu: $Q_1 = 100$ đơn vị
- Lượng cầu mới: $Q_2 = 80$ đơn vị
- Yêu cầu: Độ co giãn của cầu theo giá ($\epsilon_{Q,P}$)

(b) Gọi tên công thức hoặc định luật phù hợp:
Công thức trung điểm cho độ co giãn của cầu theo giá:
$$
\epsilon_{Q,P} = \frac{\frac{Q_2 - Q_1}{(Q_2 + Q_1)/2}}{\frac{P_2 - P_1}{(P_2 + P_1)/2}}
$$

(c) Thay số, biến đổi, rút gọn, kiểm tra sai số:
$$
\begin{aligned}
\epsilon_{Q,P} &= \frac{\frac{80 - 100}{(80 + 100)/2}}{\frac{2,50 - 2,00}{(2,50 + 2,00)/2}} \\
&= \frac{\frac{-20}{90}}{\frac{0,50}{2,25}} \\
&= \frac{-20/90}{0,50/2,25} \\
&= \frac{-20 \times 2,25}{90 \times 0,50} \\
&= \frac{-45}{45} \\
&= -1
\end{aligned}
$$

(d) Tính ra kết quả cuối cùng (dạng số hoặc biểu thức):
$\epsilon_{Q,P} = -1$

(e) So sánh kết quả thu được với từng lựa chọn:
- choice[0] → A: -0,5
- choice[1] 

In [14]:
results

[{'qid': 'val_0004', 'answer': 'B'},
 {'qid': 'val_0005', 'answer': 'C'},
 {'qid': 'val_0012', 'answer': 'B'},
 {'qid': 'val_0013', 'answer': 'D'},
 {'qid': 'val_0016', 'answer': 'B'},
 {'qid': 'val_0023', 'answer': 'A'},
 {'qid': 'val_0024', 'answer': 'A'},
 {'qid': 'val_0026', 'answer': 'B'},
 {'qid': 'val_0027', 'answer': 'C'},
 {'qid': 'val_0033', 'answer': 'C'},
 {'qid': 'val_0038', 'answer': 'B'},
 {'qid': 'val_0040', 'answer': 'D'},
 {'qid': 'val_0042', 'answer': 'B'},
 {'qid': 'val_0043', 'answer': 'D'},
 {'qid': 'val_0047', 'answer': 'A'},
 {'qid': 'val_0049', 'answer': 'B'},
 {'qid': 'val_0051', 'answer': 'A'},
 {'qid': 'val_0059', 'answer': 'A'},
 {'qid': 'val_0064', 'answer': 'A'},
 {'qid': 'val_0065', 'answer': 'B'},
 {'qid': 'val_0066', 'answer': 'E'},
 {'qid': 'val_0069', 'answer': 'A'},
 {'qid': 'val_0071', 'answer': 'A'},
 {'qid': 'val_0076', 'answer': 'D'},
 {'qid': 'val_0078', 'answer': 'A'},
 {'qid': 'val_0082', 'answer': 'B'},
 {'qid': 'val_0084', 'answer': 'D'},
 

In [ ]:
system_prompt = '''
Bạn là mô hình chuyên gia giải các bài toán STEM (Toán, Lý, Hóa, Sinh, Công nghệ, Kỹ thuật).

Nhiệm vụ cho MỖI câu hỏi:
1. Đọc nội dung câu hỏi.
2. Đọc danh sách choices (mảng không có A/B/C/D).
3. Tự gán nhãn cho choices theo vị trí:
      - choice[0] → A
      - choice[1] → B
      - choice[2] → C
      - choice[3] → D
      - ...
4. THỰC HIỆN CHAIN-OF-THOUGHT (Suy luận nội bộ), bao gồm:
      (a) Viết công thức tổng quát cần dùng.
      (b) Thay số cụ thể vào công thức.
      (c) Rút gọn kết quả về dạng chuẩn (ví dụ: k·m0·c).
      (d) So sánh kết quả thu được với từng lựa chọn.
      (e) Xác định vị trí lựa chọn đúng → A/B/C/D/E/…

Bạn được phép trình bày chain-of-thought tự do.

🚨 NHƯNG: PHẦN KẾT LUẬN CUỐI CÙNG CHO MỖI CÂU HỎI PHẢI THEO ĐÚNG REGEX SAU:

       ^ANSWER\[(?<qid>[A-Za-z0-9_]+)\]=(?<ans>[A-Z])$

Quy định output:
- Với mỗi câu hỏi, chỉ được trả về 1 dòng theo đúng format:
      ANSWER[qid]=A
- Trong đó:
      qid  là mã câu hỏi do user cung cấp.
      A/B/C/D/E/... là ký tự tương ứng vị trí lựa chọn đúng.
- answer phải PHÙ HỢP với kết quả bạn đã tính toán.
- Không được ghi nội dung đáp án, chỉ ký tự A–Z.

Để tránh lẫn với chain-of-thought, TẤT CẢ đáp án phải được đặt trong block:

---BEGIN ANSWERS---
ANSWER[q1]=X
ANSWER[q2]=Y
...
---END ANSWERS---

Yêu cầu bắt buộc:
- Không được đưa thêm bất cứ text nào vào trong block ANSWERS ngoài các dòng ANSWER[qid]=X.
- Không được bỏ sót qid nào.
- Không được thay đổi qid.
'''


In [34]:
i = 5
question_str = f"{stem_data[i]['qid']}. {stem_data[i]['question']} \n {stem_data[i]['choices']}"

user_prompt = f'''
        Câu hỏi:
        {question_str}
        '''
headers = { 
        'Authorization': vnpt_model_large["authorization"], 
        'Token-id': vnpt_model_large["tokenId"], 
        'Token-key': vnpt_model_large["tokenKey"], 
        'Content-Type': 'application/json', 
    }
json_data = {
        'model': 'vnptai_hackathon_large', 
        'messages': [
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': user_prompt},
        ],
        'temperature': 0.0, 
        'top_p': 1.0, 
        'top_k': 0, 
        'n': 1, 
        'max_completion_tokens': 512
    }
endpoint = "/v1/chat/completions/vnptai-hackathon-large"
response = requests.post(f'https://api.idg.vnpt.vn/data-service{endpoint}', headers=headers, json=json_data) 
response_js = response.json()
print(response_js["choices"][0]["message"]["content"])
result = json.loads(response_js["choices"][0]["message"]["content"])
results.extend(result)

Để tìm động lượng tương đối \( p \) của hạt, ta sử dụng công thức:

\[ p = \frac{m_0 v}{\sqrt{1 - \frac{v^2}{c^2}}} \]

trong đó:
- \( m_0 \) là khối lượng nghỉ của hạt,
- \( v \) là tốc độ của hạt,
- \( c \) là tốc độ ánh sáng.

Thay số cụ thể vào công thức với \( v = 0.6c \):

\[ p = \frac{m_0 \cdot 0.6c}{\sqrt{1 - \frac{(0.6c)^2}{c^2}}} \]

Rút gọn biểu thức dưới căn:

\[ p = \frac{m_0 \cdot 0.6c}{\sqrt{1 - 0.36}} \]
\[ p = \frac{m_0 \cdot 0.6c}{\sqrt{0.64}} \]
\[ p = \frac{m_0 \cdot 0.6c}{0.8} \]
\[ p = 0.75m_0c \]

So sánh kết quả vừa tính với từng choice:

- Choice [0]: \( 0.6m_0c \)
- Choice [1]: \( 1.25m_0c \)
- Choice [2]: \( 0.75m_0c \) (khớp với kết quả tính)
- Choice [3]: \( 1.5m_0c \)
- Choice [4]: \( 1.0m_0c \)
- Choice [5]: \( 1.2m_0c \)
- Choice [6]: \( 1.33m_0c \)
- Choice [7]: \( 1.1m_0c \)
- Choice [8]: \( 1.4m_0c \)
- Choice [9]: \( 1.6m_0c \)

Xác định lựa chọn đúng theo vị trí trong mảng: C

Đáp án: C


JSONDecodeError: Expecting value: line 1 column 1 (char 0)